# 1. Introducción

**Problema industrial:** Monitoreo de condición de bomba de alimentación al molino.

**Activo analizado:** Bomba centrífuga PUMP101 (alimentación molino).

**Origen de datos:** Exportación histórica simulada desde AVEVA PI Data Archive.

**Objetivo del análisis:** Familiarizarse con el formato de exportación PI, filtrar por calidad y analizar tendencias de temperatura de rodamiento.


# 2. Carga de librerías

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

LAB_DIR = Path.cwd()
os.chdir(LAB_DIR)
OUTPUT_DIR = LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
EXCEL_DIR = LAB_DIR / "excel"
DATA_PATH = LAB_DIR / "data" / "datos_exportados_PI.csv"


# 3. Lectura de datos PI System

Simulamos una exportación del historiador PI con columnas: `Timestamp`, `Tag`, `Value`, `Unit`, `Quality`.

In [ ]:
df_pi = pd.read_csv(DATA_PATH, parse_dates=["Timestamp"])
print(f"Registros cargados: {len(df_pi):,}")
df_pi.head(10)


# 4. Exploración del dato

In [ ]:
print("Columnas:", df_pi.columns.tolist())
print("\nEstadísticas por tag:")
display(df_pi.groupby("Tag")["Value"].describe())

calidad = df_pi["Quality"].value_counts(normalize=True) * 100
print("\nCalidad del dato (%):")
print(calidad.round(2))

faltantes = df_pi["Value"].isna().sum()
print(f"\nValores faltantes: {faltantes}")

df_good = df_pi[df_pi["Quality"] == "GOOD"].copy()
tendencia = df_good.groupby("Tag")["Value"].agg(["mean", "std", "min", "max"])
print("\nTendencia central por tag:")
display(tendencia)


# 5. Análisis matemático

Filtrado por calidad, resampleo horario y estadísticas de proceso.

In [ ]:
tag_temp = "PUMP101.BEARING_TEMP"
df_temp = df_good[df_good["Tag"] == tag_temp].set_index("Timestamp")["Value"]
serie_h = df_temp.resample("1h").mean().dropna()

media_proceso = serie_h.mean()
desv_proceso = serie_h.std()
p95 = serie_h.quantile(0.95)

resultados_export = pd.DataFrame({
    "Metrica": ["Media", "Desv_Est", "P95", "Registros_GOOD"],
    "Valor": [media_proceso, desv_proceso, p95, len(df_temp)],
    "Unidad": ["°C", "°C", "°C", "conteo"],
})
display(resultados_export)


# 6. Visualizaciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(serie_h.index, serie_h.values, color="steelblue", linewidth=1)
axes[0].axhline(media_proceso, color="red", linestyle="--", label=f"Media={media_proceso:.1f}°C")
axes[0].set_title("Tendencia PUMP101.BEARING_TEMP")
axes[0].set_xlabel("Fecha")
axes[0].set_ylabel("°C")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

df_pi["Quality"].value_counts().plot(kind="bar", ax=axes[1], color=["green", "orange"])
axes[1].set_title("Distribución de calidad PI")
axes[1].set_ylabel("Conteo")


# 7. Exportación

In [ ]:
resultados_path = OUTPUT_DIR / "resultado_analisis.csv"
graficos_path = OUTPUT_DIR / "graficos.png"
excel_resultado = EXCEL_DIR / "modelo_resultado.xlsx"

resultados_export.to_csv(resultados_path, index=False)
with pd.ExcelWriter(excel_resultado, engine="openpyxl") as writer:
    resultados_export.to_excel(writer, sheet_name="Resumen", index=False)
    tendencia.reset_index().to_excel(writer, sheet_name="Tendencia_Tags", index=False)

plt.tight_layout()
plt.savefig(graficos_path, dpi=150, bbox_inches="tight")
print(f"CSV exportado: {resultados_path}")
print(f"Gráficos exportados: {graficos_path}")
print(f"Excel exportado: {excel_resultado}")


# 8. Interpretación ingenieril

## Interpretación para mantenimiento

La temperatura de rodamiento se mantiene en rango operativo con incremento gradual en el último tercio del periodo. Se recomienda validar lubricación y alinear inspección de termografía en la próxima parada programada.
